# Part II: Stereo Vision and 3D Reconstruction

This notebook computes the depth map for 100 frames of the Kagaru Airborne dataset and back-projects them into 3D point clouds.

In [2]:
import cv2
import numpy as np
import glob
import os
import matplotlib.pyplot as plt

## 1. Camera Parameters Initialization
Here we define the intrinsic matrices (`K`), distortion coefficients (`D`), and extrinsic parameters (Rotation `R` and Translation `T`) extracted from the Kagaru dataset documentation.

In [3]:
# Intrinsic parameters for Front (Left) camera
K_front = np.array([[1641.99751, 0, 642.15139],
                    [0, 1642.30964, 470.34929],
                    [0, 0, 1]], dtype=np.float64)
D_front = np.array([-0.19978, 0.13511, -0.00007, -0.00005, 0.00000], dtype=np.float64)

# Intrinsic parameters for Back (Right) camera
K_back = np.array([[1646.07299, 0, 620.74483],
                   [0, 1645.39302, 477.47527],
                   [0, 0, 1]], dtype=np.float64)
D_back = np.array([-0.20465, 0.18856, -0.00111, 0.00040, 0.00000], dtype=np.float64)

# Extrinsic parameters (from Front to Back)
# Translation in mm is converted to meters for standard metric depth (Z will be in meters)
T = np.array([[6.09478], [-775.96641], [7.36704]], dtype=np.float64) / 1000.0

# Rotation matrix
R = np.array([[0.9995, -0.0035, 0.0302],
              [0.0031, 0.9999, 0.0112],
              [-0.0302, -0.0111, 0.9995]], dtype=np.float64)

# Image dimensions for the Pt Grey Flea2 cameras
image_size = (1280, 960)

print(f"Cameras initialized. Baseline is approx: {np.linalg.norm(T):.3f} meters")

Cameras initialized. Baseline is approx: 0.776 meters


## 2. Stereo Rectification
In this step, we compute the rectification transformations (`R1`, `R2`, `P1`, `P2`) and the disparity-to-depth mapping matrix (`Q`). Rectification aligns the images so that corresponding points lie on the same horizontal scanline (epipolar lines are horizontal), converting the 2D correspondence search into a 1D search.

In [4]:
# Perform stereo rectification
R1, R2, P1, P2, Q, validPixROI1, validPixROI2 = cv2.stereoRectify(
    cameraMatrix1=K_front, distCoeffs1=D_front,
    cameraMatrix2=K_back, distCoeffs2=D_back,
    imageSize=image_size, R=R, T=T,
    flags=cv2.CALIB_ZERO_DISPARITY, alpha=0
)

# Compute the undistortion and rectification transformation maps
# These maps tell cv2.remap how to shift pixels to rectify the images
map1_x, map1_y = cv2.initUndistortRectifyMap(K_front, D_front, R1, P1, image_size, cv2.CV_32FC1)
map2_x, map2_y = cv2.initUndistortRectifyMap(K_back, D_back, R2, P2, image_size, cv2.CV_32FC1)

print("Q Matrix (Disparity-to-Depth Mapping):\n", Q)

Q Matrix (Disparity-to-Depth Mapping):
 [[ 1.00000000e+00  0.00000000e+00  0.00000000e+00 -6.30971214e+02]
 [ 0.00000000e+00  1.00000000e+00  0.00000000e+00 -4.80312931e+02]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.66843091e+03]
 [ 0.00000000e+00  0.00000000e+00  1.28861776e+00 -0.00000000e+00]]


## 3. Depth Map Sequence and Point Cloud Generation
We configure the StereoSGBM (Semi-Global Block Matching) algorithm. We then loop through the frames, compute the disparity, visualize it into a video sequence, and back-project the depth maps into 3D point clouds (`.ply` format).

In [5]:
# Configure Semi-Global Block Matching (SGBM)
# Parameters are tuned for typical aerial/outdoor scenes
window_size = 5
min_disp = 0
num_disp = 16 * 6  # Must be divisible by 16. Adjust based on expected depth range.

stereo = cv2.StereoSGBM_create(
    minDisparity=min_disp,
    numDisparities=num_disp,
    blockSize=window_size,
    P1=8 * 3 * window_size**2,
    P2=32 * 3 * window_size**2,
    disp12MaxDiff=1,
    uniquenessRatio=10,
    speckleWindowSize=100,
    speckleRange=32,
    mode=cv2.STEREO_SGBM_MODE_SGBM_3WAY
)

# Helper function to save Point Cloud to PLY format
def write_ply(filename, points, colors):
    # Reshape points and colors
    points = points.reshape(-1, 3)
    colors = colors.reshape(-1, 3)
    
    # Filter out points with infinite/invalid depth
    mask = (points[:, 2] > 0) & (points[:, 2] < 500) # Only keep Z > 0 and Z < 500 meters
    points = points[mask]
    colors = colors[mask]
    
    # Convert BGR to RGB for standard PLY viewing
    colors = cv2.cvtColor(colors.reshape(-1, 1, 3), cv2.COLOR_BGR2RGB).reshape(-1, 3)
    
    # Construct PLY header
    header = f'''ply
format ascii 1.0
element vertex {len(points)}
property float x
property float y
property float z
property uchar red
property uchar green
property uchar blue
end_header
'''
    with open(filename, 'w') as f:
        f.write(header)
        for i in range(len(points)):
            f.write(f"{points[i,0]:.4f} {points[i,1]:.4f} {points[i,2]:.4f} {colors[i,0]} {colors[i,1]} {colors[i,2]}\n")


In [5]:
front_images_dir = r'.\100831_155323_MultiCamera0_subset_db\camera0'
back_images_dir = r'.\100831_155323_MultiCamera0_subset_db\camera1'  
front_paths = sorted(glob.glob(os.path.join(front_images_dir, '*.png')))
back_paths = sorted(glob.glob(os.path.join(back_images_dir, '*.png')))

num_frames = min(100, len(front_paths))

if num_frames == 0:
    print("WARNING: No images found. Please check your directory paths.")

# Prepare Video Writer for Depth Sequence (Grayscale visualization)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
# We will save the video as grayscale, so isColor=False
out_video = cv2.VideoWriter('depth_sequence.mp4', fourcc, 10.0, image_size, False)

# Prepare directory for Point Clouds
os.makedirs('point_clouds', exist_ok=True)

print(f"Processing {num_frames} frames...")

for i in range(num_frames):
    # Read grayscale images (for disparity) and color images (for point cloud coloring)
    img_front_gray = cv2.imread(front_paths[i], cv2.IMREAD_GRAYSCALE)
    img_back_gray = cv2.imread(back_paths[i], cv2.IMREAD_GRAYSCALE)
    img_front_color = cv2.imread(front_paths[i], cv2.IMREAD_COLOR)
    
    if img_front_gray is None or img_back_gray is None:
        print(f"Skipping frame {i} - File not read correctly")
        continue
        
    # 1. Remap images to rectify them
    rect_front = cv2.remap(img_front_gray, map1_x, map1_y, cv2.INTER_LINEAR)
    rect_back = cv2.remap(img_back_gray, map2_x, map2_y, cv2.INTER_LINEAR)
    rect_color = cv2.remap(img_front_color, map1_x, map1_y, cv2.INTER_LINEAR)
    
    # 2. Compute Disparity
    # Disparity is computed in 16ths of a pixel, so we divide by 16.0
    disparity = stereo.compute(rect_front, rect_back).astype(np.float32) / 16.0
    
    # Normalize disparity for video visualization (0 to 255)
    disp_vis = cv2.normalize(disparity, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_8U)
    out_video.write(disp_vis)
    
    # 3. Back-project to 3D Point Cloud using Q Matrix
    points_3D = cv2.reprojectImageTo3D(disparity, Q)
    
    # Save the point cloud to PLY format
    # This will generate 100 .ply files representing the metric 3D structure of the environment
    write_ply(f"point_clouds/frame_{i:03d}.ply", points_3D, rect_color)
    
    if (i+1) % 10 == 0:
        print(f"Processed {i+1}/{num_frames} frames")

out_video.release()
if num_frames > 0:
    print("Processing complete. Video saved as 'depth_sequence.mp4' and Point Clouds saved in 'point_clouds' folder.")


Processing 100 frames...
Processed 10/100 frames
Processed 20/100 frames
Processed 30/100 frames
Processed 40/100 frames
Processed 50/100 frames
Processed 60/100 frames
Processed 70/100 frames
Processed 80/100 frames
Processed 90/100 frames
Processed 100/100 frames
Processing complete. Video saved as 'depth_sequence.mp4' and Point Clouds saved in 'point_clouds' folder.


In [7]:
# save color full
# Prepare Video Writer for Depth Sequence (Color visualization)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_video = cv2.VideoWriter('depth_sequence_color.mp4', fourcc, 10.0, image_size, True)

# Prepare directory for Point Clouds
os.makedirs('point_clouds', exist_ok=True)

print(f"Processing {num_frames} frames...")

for i in range(num_frames):
    # Read grayscale images (for disparity) and color images (for point cloud coloring)
    img_front_gray = cv2.imread(front_paths[i], cv2.IMREAD_GRAYSCALE)
    img_back_gray = cv2.imread(back_paths[i], cv2.IMREAD_GRAYSCALE)
    img_front_color = cv2.imread(front_paths[i], cv2.IMREAD_COLOR)
    
    if img_front_gray is None or img_back_gray is None:
        print(f"Skipping frame {i} - File not read correctly")
        continue
        
    # 1. Remap images to rectify them
    rect_front = cv2.remap(img_front_gray, map1_x, map1_y, cv2.INTER_LINEAR)
    rect_back = cv2.remap(img_back_gray, map2_x, map2_y, cv2.INTER_LINEAR)
    rect_color = cv2.remap(img_front_color, map1_x, map1_y, cv2.INTER_LINEAR)
    
    # 2. Compute Disparity
    disparity = stereo.compute(rect_front, rect_back).astype(np.float32) / 16.0
    
    # Normalize disparity for video visualization (0 to 255)
    disp_vis = cv2.normalize(disparity, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_8U)
    
    disp_color = cv2.applyColorMap(disp_vis, cv2.COLORMAP_JET)
    
    out_video.write(disp_color)
    # ------------------------------------------
    
    # 3. Back-project to 3D Point Cloud using Q Matrix
    points_3D = cv2.reprojectImageTo3D(disparity, Q)
    
    # Save the point cloud to PLY format
    write_ply(f"point_clouds/frame_{i:03d}.ply", points_3D, rect_color)
    
    if (i+1) % 10 == 0:
        print(f"Processed {i+1}/{num_frames} frames")

out_video.release()
if num_frames > 0:
    print("Processing complete. Video saved as 'depth_sequence_color.mp4' and Point Clouds saved in 'point_clouds' folder.")

Processing 100 frames...
Processed 10/100 frames
Processed 20/100 frames
Processed 30/100 frames
Processed 40/100 frames
Processed 50/100 frames
Processed 60/100 frames
Processed 70/100 frames
Processed 80/100 frames
Processed 90/100 frames
Processed 100/100 frames
Processing complete. Video saved as 'depth_sequence_color.mp4' and Point Clouds saved in 'point_clouds' folder.


In [8]:
#generate main frames(left side) video
# --- Generate Original Color Sequence Video ---
# This cell uses the variables already in memory (front_paths, map1_x, map1_y, image_size, num_frames)

print(f"Generating original rectified video for {num_frames} frames...")

# 1. Setup Video Writer for the original sequence
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
# fps is 10.0, same as the depth sequence
out_video_original = cv2.VideoWriter('original_sequence.mp4', fourcc, 10.0, image_size, True)

for i in range(num_frames):
    # 2. Read the color image from the front camera
    img_front_color = cv2.imread(front_paths[i], cv2.IMREAD_COLOR)
    
    if img_front_color is None:
        print(f"Skipping frame {i} - File not read correctly")
        continue
        
    # 3. Rectify the color image so it spatially matches the depth map perfectly
    rect_color = cv2.remap(img_front_color, map1_x, map1_y, cv2.INTER_LINEAR)
    
    # 4. Write to video
    out_video_original.write(rect_color)
    
    if (i+1) % 10 == 0:
        print(f"Processed {i+1}/{num_frames} color frames")

out_video_original.release()
print("Process complete! The video is saved as 'original_sequence.mp4'.")

Generating original rectified video for 100 frames...
Processed 10/100 color frames
Processed 20/100 color frames
Processed 30/100 color frames
Processed 40/100 color frames
Processed 50/100 color frames
Processed 60/100 color frames
Processed 70/100 color frames
Processed 80/100 color frames
Processed 90/100 color frames
Processed 100/100 color frames
Process complete! The video is saved as 'original_sequence.mp4'.


In [8]:
import cv2
import os
import numpy as np

output_dir = 'report_samples_kagaru'
os.makedirs(output_dir, exist_ok=True)

sample_indices = [2, 50, 100] 

print("Generating 3 sample image pairs for the report...")

for idx, frame_idx in enumerate(sample_indices):
    if frame_idx >= len(front_paths):
        print(f"Frame {frame_idx} is out of bounds. Skipping.")
        continue
        
    img_left = cv2.imread(back_paths[frame_idx], cv2.IMREAD_COLOR)
    img_right = cv2.imread(front_paths[frame_idx], cv2.IMREAD_COLOR)
    
    img_left_gray = cv2.cvtColor(img_left, cv2.COLOR_BGR2GRAY)
    img_right_gray = cv2.cvtColor(img_right, cv2.COLOR_BGR2GRAY)
    
    disparity = stereo.compute(img_left_gray, img_right_gray).astype(np.float32) / 16.0
    
    disp_vis_input = np.maximum(disparity, 0)
    disp_vis = cv2.normalize(disp_vis_input, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_8U)
    disp_color = cv2.applyColorMap(disp_vis, cv2.COLORMAP_JET)
    
    orig_name = os.path.join(output_dir, f"orig_sample_{idx+1}.jpg")
    depth_name = os.path.join(output_dir, f"depth_sample_{idx+1}.jpg")
    
    cv2.imwrite(orig_name, img_left)
    cv2.imwrite(depth_name, disp_color)
    
    print(f"Saved Sample {idx+1}: {orig_name} & {depth_name}")

print("All sample images saved successfully in the 'report_samples' folder.")

Generating 3 sample image pairs for the report...
Saved Sample 1: report_samples_kagaru\orig_sample_1.jpg & report_samples_kagaru\depth_sample_1.jpg
Saved Sample 2: report_samples_kagaru\orig_sample_2.jpg & report_samples_kagaru\depth_sample_2.jpg
Saved Sample 3: report_samples_kagaru\orig_sample_3.jpg & report_samples_kagaru\depth_sample_3.jpg
All sample images saved successfully in the 'report_samples' folder.
